# Проверка сервисов BM25 и HNSW

Два сервиса на сервере, оба отвечают списком треков (`ids`) с оценками (`scores`):

* **BM25** (ветка `dev/bm25`) — поиск по словам: жанры, теги Last.fm. Слова **английские**, как в каталоге.
* **HNSW** (релиз `hnsw-v1`) — текстовый семантический поиск: запрос как есть переводится в вектор
  EmbeddingGemma-2 и сравнивается с текстами треков (метаданные, теги, описание, лирика).

Сами они ничего не рекомендуют: это кандидаты, из которых пайплайн собирает ответ (`inference.ipynb`).
Описание API: `docs/candgen_api.md`.

Адреса и ключи — `BM25_URL`, `BM25_API_KEY`, `HNSW_URL`, `HNSW_API_KEY` из файла `.env` в корне репозитория или секретов
DataSphere. Те же запросы можно делать в браузере: `<адрес>/docs` (ключ — в поле `x-api-key`).

## Настройки

`DATA_DIR` — где лежит `tracks_meta`, из которого строились индексы (отсюда берутся названия треков).

In [ ]:
BM25_URL = "${BM25_URL}"    # из .env / окружения; можно вписать адрес прямо сюда
HNSW_URL = "${HNSW_URL}"
DATA_DIR = "music4all_crs"  # датасет: отсюда названия треков

In [ ]:
import os
import pandas as pd

if os.path.basename(os.getcwd()) == "examples":
    os.chdir("..")
from recsys.data.crs import split_path
from recsys.config import load_env
from recsys.retrieval.remote import CandgenClient

load_env()  # адреса и ключи из .env
bm25 = CandgenClient(BM25_URL, api_key="${BM25_API_KEY}")
hnsw = CandgenClient(HNSW_URL, api_key="${HNSW_API_KEY}", timeout=30)
print("BM25:", bm25.url, "| HNSW:", hnsw.url)
columns = ["m4a_id", "m4a_artist", "m4a_song", "m4a_genres_full"]
tracks = pd.read_parquet(split_path(DATA_DIR, "tracks_meta"), columns=columns).set_index("m4a_id")


def show(answer):
    """Ответ сервиса -> таблица: артист, песня, жанры, оценка."""
    return tracks.reindex(answer["ids"]).assign(score=answer["scores"])

## 1. Сервисы работают?

Какие индексы загружены и сколько в них треков. Ошибка `Connection refused` — сервис не запущен.

In [ ]:
pd.DataFrame(bm25.get("/health")["indexes"]).T

In [ ]:
hnsw.get("/health")

## 2. BM25: поиск по словам

* `words` — английские слова, регистр не важен
* `k` — сколько треков вернуть (до 1000)
* `index` — где искать: на сервере один индекс `cards` по всей карточке трека (`genres`, `tags` — его старые имена)

Чем больше слов запроса у трека, тем выше оценка.

In [ ]:
show(bm25.post("/bm25/search", {"words": ["rock"], "k": 5, "index": "cards"}))

In [ ]:
show(bm25.post("/bm25/search", {"words": ["hard", "rock"], "k": 5, "index": "cards"}))

`exclude_ids` — какие треки не возвращать (пайплайн передаёт сюда уже показанные).

In [ ]:
first = bm25.post("/bm25/search", {"words": ["rock"], "k": 5, "index": "cards"})["ids"][0]
show(bm25.post("/bm25/search", {"words": ["rock"], "k": 5, "index": "cards", "exclude_ids": [first]}))

BM25 ищет только точные слова из тегов: фразу целиком надо разбить на слова, а слов, которых нет в тегах, он не знает.
Какие слова из запроса уходят в BM25, решает разбор запроса (`inference.ipynb`, раздел 5). HNSW понимает запрос
целиком — следующий раздел.

In [ ]:
bm25.post("/bm25/search", {"words": ["melancholic evening drive"], "k": 5, "index": "cards"})  # фраза — одно «слово»: не найдётся

## 3. HNSW: поиск по смыслу запроса

* `query` — текст запроса как есть; профиль пользователя добавлять не нужно (по замерам автора — хуже)
* `k`, `exclude_ids` — как у BM25; поля `index` нет

Оценка — косинусная близость запроса и текста трека. Сильнее всего на точных названиях и строчках из песен, слабее —
на настроении и ситуации.

In [ ]:
show(hnsw.post("/hnsw/search", {"query": "calm jazz for a late evening", "k": 5}))

In [ ]:
show(hnsw.post("/hnsw/search", {"query": "Nirvana Smells Like Teen Spirit", "k": 5}))

In [ ]:
show(hnsw.post("/hnsw/search", {"query": "a song about rain and being alone", "k": 5}))

## 4. Ошибки

| ответ | причина |
|---|---|
| `HTTP 404 Unknown index` | нет такого индекса |
| `HTTP 401` | нет ключа или он неверный (`BM25_API_KEY`, `HNSW_API_KEY`; ключи у сервисов разные) |
| `HTTP 422` | ошибка в теле запроса: лишнее поле (у HNSW — `index`), `k` > 1000 |
| `Connection refused`, таймаут | сервис не запущен или недоступен по сети |

In [ ]:
from recsys.retrieval.remote import CandgenError

try:
    bm25.post("/bm25/search", {"words": ["rock"], "k": 5, "index": "lyrics"})
except CandgenError as e:
    print(e)